In [275]:
import pandas as pd
import numpy as np
import random
from copy import deepcopy

school = 'school_1'

# Define paths
raw_data_folder = '../data/raw_data'
processed_data_folder = '../data/processed_data'
processed_data_folder_school = f'{processed_data_folder}/{school}'
print(f'Processed data folder: {processed_data_folder_school}')

# Read in processed data
constraints_students = pd.read_csv(f'{processed_data_folder_school}/constraints_students.csv')
constraints_teachers = pd.read_csv(f'{processed_data_folder_school}/constraints_teachers.csv')
current_groups = pd.read_csv(f'{processed_data_folder_school}/current_groups.csv')
group_preferences = pd.read_csv(f'{processed_data_folder_school}/group_preferences.csv')
info_students = pd.read_csv(f'{processed_data_folder_school}/info_students.csv')
info_teachers = pd.read_csv(f'{processed_data_folder_school}/info_teachers.csv')

n_students, n_groups, min_group_size, max_extra_care_1, max_extra_care_2 = group_preferences.iloc[0]

Processed data folder: ../data/processed_data/school_1


In [276]:
def get_max_group_size(min_group_size, n_students, n_groups):
    remaining = n_students - ( min_group_size * n_groups)
    max_size = min_group_size + (remaining // n_groups)
    if remaining % n_groups != 0:
        max_size += 1
    return max_size

max_group_size = get_max_group_size(min_group_size, n_students, n_groups)

In [277]:
def get_assigned_students(groups):
    return [student for group in groups.values() for student in group]

def is_assigned(student, groups):
    if student in get_assigned_students(groups):
        return True
    return False

def get_group(student, groups):
    group = [group for group in groups if student in groups[group]]
    return group[0]

In [278]:
# First assign students that can only have one option
def assign_includes(groups, constraints_teachers, constraints_students):
    teachers_include = constraints_teachers[constraints_teachers['Together'] == 'Yes']
    students_include = constraints_students[constraints_students['Together'] == 'Yes']

    # Assign students to teachers based on inclusion constraints
    for _, (student, teacher, _) in teachers_include.iterrows():
        groups[teacher].append(student)

    # Add students that must be together to the same group if already assigned
    for _, (s1, s2, _) in students_include.iterrows():
        if (is_assigned(s1, groups) and is_assigned(s2, groups)) or (not is_assigned(s1, groups) and not is_assigned(s2, groups)):
            # If both students are already assigned or not assigned, do nothing
            continue
        elif is_assigned(s1, groups):
            group = get_group(s1, groups)
            groups[group].append(s2)
        elif is_assigned(s2, groups):
            group = get_group(s2, groups)
            groups[group].append(s1)

    return groups

In [279]:
# This function only checks max assignments of a certain binary column - if there are minimum constraints an extra check is needed
def violates_binary(student, group, groups, student_info, col, limit):
   value = student_info.loc[student_info['Student'] == student, col].values[0]
   if value == 'No':
      return False

   # Get number of binary students in the group
   count = student_info[student_info['Student'].isin(groups[group]) & (student_info[col] == 'Yes')].shape[0]

   # Check if the number of binary students in the group exceeds the limit
   if count >= limit:
      return True

   return False

def violates_min_group_size(group, groups, min_group_size):
   # Check if the group size is less than the minimum group size
   if len(groups[group]) < min_group_size:
      return True
   return False

def violates_max_group_size(group, groups, max_group_size):
      if len(groups[group]) >= max_group_size:
         return True
      return False


In [280]:
def violates_student_pair(student, group, groups, constraints_students, assigned_students):
    # Check if student is in constraints
    mask = (constraints_students['Student 1'] == student) | (constraints_students['Student 2'] == student)
    if not mask.any():
        return False

    # Get the other students
    matches = constraints_students[mask].copy()
    matches['Other'] = matches.apply(lambda row: row['Student 2'] if row['Student 1'] == student else row['Student 1'], axis=1)

    includes = matches[matches['Together'] == 'Yes']['Other'].tolist()
    excludes = matches[matches['Together'] != 'Yes']['Other'].tolist()

    # Check if all includes are in the group
    for include in includes:
        if include in assigned_students and include not in groups[group]:
            return True

    # Check if any excludes are in the group
    for exclude in excludes:
        if exclude in assigned_students and exclude in groups[group]:
            return True

    return False

In [281]:
def violates_teacher_pair(student, group, constraints_teachers):
    # Check if student teacher pair is in constraint
    match = constraints_teachers[(constraints_teachers['Student'] == student) & (constraints_teachers['Teacher'] == group)]

    if match.empty:
        return False

    # Check if they are allowed to be together
    if match['Together'].values[0] == 'No':
        return True

    return False

In [282]:
def random_assign_student(groups, unassigned_students, assigned_students, info_students,  constraints_students, constraints_teachers, max_group_size, max_extra_care_1):
    for student in unassigned_students:
        random_groups = list(groups.keys())
        random.shuffle(random_groups)

        for group in random_groups:
            if violates_max_group_size(group, groups, max_group_size):
                # print(f"WARNING: Group {group} is already full.")
                continue
            if violates_binary(student, group, groups, info_students, 'Extra Care', max_extra_care_1):
                # print(f"WARNING: Group {group} has too many students with Extra Care.")
                continue
            # if violates_binary(student, group, groups, info_students, 'Extra Care 2', max_extra_care_2):
            #     continue
            if violates_student_pair(student, group, groups, constraints_students, assigned_students):
                # print(f"WARNING: Student {student} violates student pair constraints in group {group}.")
                continue
            if violates_teacher_pair(student, group, constraints_teachers):
                # print(f"WARNING: Student {student} violates teacher pair constraints in group {group}.")
                continue

            groups[group].append(student)
            assigned_students.append(student)
            print(f"Assigned student {student} to group {group}.")
            break
        else:
            print(f"WARNING: Couldn't assign student {student} due to constraints.")

    return groups

In [283]:
def valid_groups(groups, info_students, constraints_students, constraints_teachers, min_group_size, max_group_size, max_extra_care_1):
    assigned_students = get_assigned_students(groups)

    for group in groups:
        if len(groups[group]) < min_group_size or len(groups[group]) > max_group_size:
            print(f"Group {group} violates size constraints.")
            return False

        # Add additional checks for other care columns if needed in the list
        # Check Extra Care limits
        for care_col, limit in [('Extra Care', max_extra_care_1)]:
            count = info_students[info_students['Student'].isin(groups[group]) & (info_students[care_col] == 'Yes')].shape[0]
            if count > limit:
                print(f' violates max constraint for {care_col} in group {group} with limit {limit}.')
                return False

        # Check student-student constraints
        for student in groups[group]:
            if violates_student_pair(student, group, groups, constraints_students,assigned_students):
                print(f"Student {student} violates student pair constraints in group {group}.")
                return False

            if violates_teacher_pair(student, group, constraints_teachers):
                print(f"Student {student} violates teacher pair constraints in group {group}.")
                return False

    # Ensure all students assigned
    if len(assigned_students) != len(info_students):
        print(f"Not all students assigned. Assigned: {len(assigned_students)}, Total: {len(info_students)}")
        return False

    return True

In [284]:
def main(seed=42, max_attempts=100):
    # Set random seed for reproducibility
    random.seed(seed)

    # Initialize groups
    base_groups = {}
    for teacher in info_teachers['Teacher']:
        base_groups[teacher] = []

    # Assign includes
    initial_groups = assign_includes(base_groups, constraints_teachers, constraints_students)
    initial_assigned_students = get_assigned_students(initial_groups)

    for i in range(max_attempts):
        groups = deepcopy(initial_groups)
        assigned_students = initial_assigned_students.copy()

        care_students = [s for s in info_students[info_students['Extra Care'] == 'Yes']['Student'] if s not in assigned_students]
        other_students = [s for s in info_students[~(info_students['Student'].isin(care_students))]['Student'] if s not in assigned_students]

        random.shuffle(care_students)
        random.shuffle(other_students)
        unassigned_students = care_students + other_students

        # Run one attempt
        groups = random_assign_student(groups, unassigned_students, assigned_students,
                                       info_students, constraints_students, constraints_teachers,
                                       max_group_size, max_extra_care_1)

        # Check if groups are valid
        if valid_groups(groups, info_students, constraints_students, constraints_teachers, min_group_size, max_group_size, max_extra_care_1):
            print(f'Valid groups found after {i+1} attempts.')
            return groups


    # # If no valid groups found after max attempts, return the last attempt
    print('No valid groups found after maximum attempts.')
    return groups

main()

Assigned student S_21 to group T_02.
Assigned student S_20 to group T_02.
Assigned student S_22 to group T_01.
Assigned student S_09 to group T_03.
Assigned student S_23 to group T_02.
Assigned student S_07 to group T_03.
Assigned student S_26 to group T_02.
Assigned student S_12 to group T_03.
Assigned student S_18 to group T_01.
Assigned student S_15 to group T_03.
Assigned student S_25 to group T_02.
Assigned student S_16 to group T_03.
Assigned student S_19 to group T_01.
Assigned student S_24 to group T_01.
Assigned student S_03 to group T_02.
Assigned student S_17 to group T_02.
Assigned student S_01 to group T_02.
Assigned student S_08 to group T_02.
Assigned student S_14 to group T_02.
Assigned student S_04 to group T_01.
Assigned student S_05 to group T_01.
Assigned student S_06 to group T_01.
Assigned student S_27 to group T_01.
Assigned student S_11 to group T_01.
Valid groups found after 1 attempts.


{'T_01': ['S_28',
  'S_22',
  'S_18',
  'S_19',
  'S_24',
  'S_04',
  'S_05',
  'S_06',
  'S_27',
  'S_11'],
 'T_02': ['S_21',
  'S_20',
  'S_23',
  'S_26',
  'S_25',
  'S_03',
  'S_17',
  'S_01',
  'S_08',
  'S_14'],
 'T_03': ['S_10',
  'S_02',
  'S_30',
  'S_29',
  'S_13',
  'S_09',
  'S_07',
  'S_12',
  'S_15',
  'S_16']}